# Recuperacion de mapeos para inferencia

Esta notebook concilia los datos procesados del modelo con las fuentes en S3 para identificar y versionar las transformaciones necesarias en inferencia.

In [16]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [17]:
from pathlib import Path
import io
import json
import os
import re

import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

AWS_REGION = "us-east-1"
BUCKET = "ibk-discovery-comercial-us-east-1-654654352211-data"
MODEL_PREFIX = "discovery/comercial/sanherna/PLAFT/PJ/MINORISTA"
BASE_S3 = f"s3://{BUCKET}/{MODEL_PREFIX}"
PERIODO = 202508

credentials_path = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")
if credentials_path.exists():
    export_pattern = re.compile(r"^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$")
    for line in credentials_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        match = export_pattern.match(line.strip())
        if match:
            key, value = match.groups()
            os.environ[key] = value.strip().strip('"').strip("'")

session = boto3.Session(region_name=AWS_REGION)
s3 = session.client("s3")
identity = session.client("sts").get_caller_identity()
print(f"S3 listo: {BASE_S3}")
print(f"AWS account: {identity['Account']}")

S3 listo: s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA
AWS account: 654654352211


## 2. Cargar headers y definir orden de columnas

In [18]:
DATA_DEV_PREFIX = f"{MODEL_PREFIX}/data_dev_model"
HEADERS_KEY = f"{DATA_DEV_PREFIX}/headers_total.csv"


def read_csv_s3(key: str, **kwargs) -> pd.DataFrame:
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(body), **kwargs)


headers = read_csv_s3(HEADERS_KEY)
if headers["variables"].isna().any() or headers["variables"].duplicated().any():
    raise ValueError("headers_total.csv contiene nombres vacios o duplicados.")

column_order = headers["variables"].tolist()
print(f"Variables del modelo: {len(column_order)}")
print(column_order[:10])

Variables del modelo: 33
['target', 'cnt_trx_cargostot_3m', 'mto_pas_soles', 'rat_pastot_x_ingtot_6m', 'cnt_trx_abonospromtot_3m', 'imp_trx_abonosefect_6m', 'num_antiguedad', 'imp_trx_cargosefe_6m', 'ratio_cargos_1m_vs_6m', 'cnt_meses_sinegresos_12m']


## 3. Cargar train, validation y test

In [19]:
dataset_keys = {
    "train": f"{DATA_DEV_PREFIX}/train_total.csv",
    "val": f"{DATA_DEV_PREFIX}/validation_total.csv",
    "test": f"{DATA_DEV_PREFIX}/test_total.csv",
}
datasets = {
    name: read_csv_s3(key, header=None, names=column_order)
    for name, key in dataset_keys.items()
}

for name, frame in datasets.items():
    if frame.columns.tolist() != column_order:
        raise ValueError(f"Orden de columnas invalido en {name}.")
    print(f"{name}: {frame.shape}")

df_train, df_val, df_test = datasets["train"], datasets["val"], datasets["test"]

train: (173600, 33)
val: (324253, 33)
test: (1509824, 33)


## 4. Anexar columnas de trazabilidad

In [20]:
trace_columns = ["key_value", "cod_cli", "cod_mes", "tipo_alerta_n2", "trx_riesgo_cliente"]
extras_val = read_csv_s3(f"{DATA_DEV_PREFIX}/extras_validation_total.csv")
extras_test = read_csv_s3(f"{DATA_DEV_PREFIX}/extras_test_total.csv")

for name, frame, extras in [("val", df_val, extras_val), ("test", df_test, extras_test)]:
    if len(frame) != len(extras):
        raise ValueError(f"Filas desalineadas entre {name} y sus extras.")
    missing = [column for column in trace_columns if column not in extras.columns]
    if missing:
        raise ValueError(f"Extras de {name} sin columnas: {missing}")

df_val = pd.concat([df_val.reset_index(drop=True), extras_val[trace_columns]], axis=1)
df_test = pd.concat([df_test.reset_index(drop=True), extras_test[trace_columns]], axis=1)
print(df_val[trace_columns + ["target"]].head())

                                           key_value     cod_cli   cod_mes  \
0  3B020DB4DC631ED9192290632AD8627B2A9495500B2783...  21602930.0  202508.0   
1  2820FF155EBE1FF50F339495F0BDE21578CCE6AFD934D9...   7321045.0  202508.0   
2  3194E86A626832BA5994B2318E2D0A0774892E5D769D76...  19639492.0  202508.0   
3  566A75C788AB628F0A7BAE537CBBBD49F613E1FEAA85A4...  21010838.0  202509.0   
4  6A5C84509F82D207D3F611E0FBBBA19AA144C80E686E3E...  18466306.0  202509.0   

  tipo_alerta_n2 trx_riesgo_cliente  target  
0       SIN_INFO           SIN_INFO     0.0  
1       SIN_INFO           SIN_INFO     0.0  
2       SIN_INFO           SIN_INFO     0.0  
3       SIN_INFO           SIN_INFO     0.0  
4       SIN_INFO           SIN_INFO     0.0  


## 5. Cargar el parquet historico `df_3`

In [21]:
DF3_KEY = f"{MODEL_PREFIX}/DATA_INFERENCIA/data_pn_total_expandido_new_v1.parquet"
RAW_PREFIX = f"{MODEL_PREFIX}/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo={PERIODO}/"

response = s3.get_object(Bucket=BUCKET, Key=DF3_KEY)
df_3 = pq.read_table(io.BytesIO(response["Body"].read())).to_pandas()
df_3 = df_3[pd.to_numeric(df_3["cod_mes"], errors="coerce") == PERIODO].copy()

raw_keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=RAW_PREFIX):
    raw_keys.extend(obj["Key"] for obj in page.get("Contents", []))
if not raw_keys:
    raise FileNotFoundError(f"No hay parquet de inferencia en {RAW_PREFIX}")

df_raw = pd.concat(
    [pq.read_table(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())).to_pandas() for key in raw_keys],
    ignore_index=True,
)
print(f"df_3 {PERIODO}: {df_3.shape}; raw inferencia: {df_raw.shape}")
print(df_3[["cod_ubigeo_cd", "cod_sectorista_id"]].dtypes)

df_3 202508: (161415, 65); raw inferencia: (161435, 60)
cod_ubigeo_cd        int64
cod_sectorista_id    int64
dtype: object


## 6. Comparar esquema y tipos

In [22]:
critical_columns = ["cod_ubigeo_cd", "cod_sectorista_id"]
schema_report = pd.DataFrame({
    "column": critical_columns,
    "df_3_dtype": [str(df_3[column].dtype) for column in critical_columns],
    "raw_dtype": [str(df_raw[column].dtype) for column in critical_columns],
    "df_3_unique": [df_3[column].nunique(dropna=False) for column in critical_columns],
    "raw_unique": [df_raw[column].nunique(dropna=False) for column in critical_columns],
    "df_3_null_pct": [df_3[column].isna().mean() * 100 for column in critical_columns],
    "raw_null_pct": [df_raw[column].isna().mean() * 100 for column in critical_columns],
})
display(schema_report)

,column,df_3_dtype,raw_dtype,df_3_unique,raw_unique,df_3_null_pct,raw_null_pct
0,cod_ubigeo_cd,int64,object,3,1198,0.0,0.0
1,cod_sectorista_id,int64,object,3,632,0.0,0.0


## 7. Auditar valores originales y valores procesados

In [23]:
df_val_periodo = df_val[pd.to_numeric(df_val["cod_mes"], errors="coerce") == PERIODO].copy()
for frame in [df_val_periodo, df_3, df_raw]:
    frame["key_value"] = frame["key_value"].astype(str).str.strip()

comparison = df_raw[["key_value"] + critical_columns].merge(
    df_val_periodo[["key_value"] + critical_columns],
    on="key_value",
    how="inner",
    suffixes=("_raw", "_processed"),
)
print(f"Filas conciliadas contra validation_total: {len(comparison):,}")

for column in critical_columns:
    display(
        comparison[[f"{column}_raw", f"{column}_processed"]]
        .value_counts()
        .head(20)
        .rename("cantidad")
        .reset_index()
    )

Filas conciliadas contra validation_total: 161,434


,cod_ubigeo_cd_raw,cod_ubigeo_cd_processed,cantidad
0,150101,3.0,9780
1,150140,3.0,7208
2,130101,3.0,5443
3,150122,3.0,5424
4,150135,3.0,5139
5,150132,3.0,4389
6,150103,3.0,4088
7,150117,3.0,3779
8,150131,3.0,3761
9,150115,3.0,3700


,cod_sectorista_id_raw,cod_sectorista_id_processed,cantidad
0,D1120,3.0,118973
1,71513,3.0,3714
2,71516,3.0,3311
3,71518,2.0,1182
4,D1B21,3.0,1052
5,D324T,1.0,815
6,71514,2.0,670
7,D1310,3.0,669
8,D141F,3.0,568
9,D1L10,3.0,527


## 8. Auditar `mto_fact_declarado_sunat`

Esta variable se analiza por separado porque llega vacía en la inferencia actual y no debe imputarse ni mapearse sin una relación verificable con su valor fuente.

In [29]:
FEATURE = "mto_fact_declarado_sunat"

fact_comparison = df_raw[["key_value", FEATURE]].merge(
    df_val_periodo[["key_value", FEATURE]],
    on="key_value",
    how="inner",
    suffixes=("_raw", "_processed"),
)
fact_comparison[f"{FEATURE}_raw"] = fact_comparison[f"{FEATURE}_raw"].fillna("SIN_INFO").astype(str)
fact_comparison[f"{FEATURE}_processed"] = pd.to_numeric(
    fact_comparison[f"{FEATURE}_processed"], errors="coerce"
)

summary = pd.DataFrame([
    {
        "filas_conciliadas": len(fact_comparison),
        "raw_null_pct": (fact_comparison[f"{FEATURE}_raw"] == "SIN_INFO").mean() * 100,
        "raw_valores_unicos": fact_comparison[f"{FEATURE}_raw"].nunique(),
        "niveles_procesados": fact_comparison[f"{FEATURE}_processed"].nunique(),
    }
])
display(summary)
display(
    fact_comparison[[f"{FEATURE}_raw", f"{FEATURE}_processed"]]
    .value_counts(dropna=False)
    .rename("cantidad")
    .reset_index()
    .sort_values("cantidad", ascending=False)
)

mapping_candidates = fact_comparison.groupby(f"{FEATURE}_raw")[f"{FEATURE}_processed"].nunique(dropna=True)
ambiguous_values = mapping_candidates[mapping_candidates > 1]
recoverable_values = mapping_candidates[mapping_candidates == 1]

print(f"Valores crudos recuperables: {len(recoverable_values):,}")
print(f"Valores crudos ambiguos: {len(ambiguous_values):,}")
if (fact_comparison[f"{FEATURE}_raw"] == "SIN_INFO").all():
    print("Resultado: la fuente de inferencia no trae valor original; no se puede construir un mapeo global.")

,filas_conciliadas,raw_null_pct,raw_valores_unicos,niveles_procesados
0,161434,100.0,1,3


,mto_fact_declarado_sunat_raw,mto_fact_declarado_sunat_processed,cantidad
0,SIN_INFO,3.0,134519
1,SIN_INFO,1.0,14148
2,SIN_INFO,2.0,12767


Valores crudos recuperables: 0
Valores crudos ambiguos: 1
Resultado: la fuente de inferencia no trae valor original; no se puede construir un mapeo global.


## 8. Construir tablas de mapeo

In [30]:
encoding_maps = {}
for column in critical_columns:
    source = f"{column}_raw"
    encoded = f"{column}_processed"
    table = comparison[[source, encoded]].dropna(subset=[source]).copy()
    table[source] = table[source].astype(str)
    conflicts = table.groupby(source)[encoded].nunique()
    conflicts = conflicts[conflicts > 1]
    if not conflicts.empty:
        raise ValueError(f"Mapeo ambiguo para {column}: {conflicts.index[:10].tolist()}")
    encoding_maps[column] = {
        key: int(value)
        for key, value in table.drop_duplicates(source).set_index(source)[encoded].items()
    }

print({column: len(mapping) for column, mapping in encoding_maps.items()})

{'cod_ubigeo_cd': 1198, 'cod_sectorista_id': 632}


## 9. Aplicar mapeo y validar cobertura

In [31]:
coverage_rows = []
for column, mapping in encoding_maps.items():
    raw_values = df_raw[column].fillna("SIN_INFO").astype(str)
    mapped = raw_values.map(mapping).fillna(0).astype(int)
    expected = comparison[f"{column}_processed"].astype(int)
    observed = comparison[f"{column}_raw"].fillna("SIN_INFO").astype(str).map(mapping).fillna(0).astype(int)
    coverage_rows.append({
        "variable": column,
        "coverage_pct": (raw_values.isin(mapping).mean() * 100),
        "unknown_rows": int((~raw_values.isin(mapping)).sum()),
        "exact_match_validation_pct": ((observed == expected).mean() * 100),
    })

coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report)
if (coverage_report["exact_match_validation_pct"] < 100).any():
    raise AssertionError("El mapa no reproduce exactamente la validacion.")

,variable,coverage_pct,unknown_rows,exact_match_validation_pct
0,cod_ubigeo_cd,100.0,0,100.0
1,cod_sectorista_id,100.0,0,100.0


## 10. Integrar el mapeo en `inference.py`

In [32]:
def apply_feature_mappings(df: pd.DataFrame, maps: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    report = []
    for column, mapping in maps.items():
        if column not in df.columns:
            raise KeyError(f"Falta la columna requerida: {column}")
        raw_values = df[column].fillna("SIN_INFO").astype(str)
        mapped = raw_values.map(mapping)
        report.append({
            "variable": column,
            "mapped_pct": mapped.notna().mean() * 100,
            "unknown_count": int(mapped.isna().sum()),
        })
        df[column] = mapped.fillna(0).astype("float32")
    return df, pd.DataFrame(report)

inference_snippet = '''
def apply_feature_mappings(df, maps):
    for column, mapping in maps.items():
        df[column] = df[column].fillna("SIN_INFO").astype(str).map(mapping).fillna(0).astype("float32")
    return df

with open(os.path.join(DIR_ARTIFACTS, "encoding_maps_recuperados.json"), encoding="utf-8") as file:
    recovered_maps = json.load(file)
df = apply_feature_mappings(df, recovered_maps)
'''
print(inference_snippet)


def apply_feature_mappings(df, maps):
    for column, mapping in maps.items():
        df[column] = df[column].fillna("SIN_INFO").astype(str).map(mapping).fillna(0).astype("float32")
    return df

with open(os.path.join(DIR_ARTIFACTS, "encoding_maps_recuperados.json"), encoding="utf-8") as file:
    recovered_maps = json.load(file)
df = apply_feature_mappings(df, recovered_maps)



## 11. Guardar artefacto y comprobar la aplicacion

In [33]:
OUTPUT_DIR = Path("artifacts_recuperados")
OUTPUT_DIR.mkdir(exist_ok=True)
map_path = OUTPUT_DIR / f"encoding_maps_recuperados_{PERIODO}.json"
coverage_path = OUTPUT_DIR / f"coverage_mapeos_{PERIODO}.csv"

with map_path.open("w", encoding="utf-8") as file:
    json.dump(encoding_maps, file, ensure_ascii=False, indent=2, sort_keys=True)
coverage_report.to_csv(coverage_path, index=False)

mapped_raw, runtime_coverage = apply_feature_mappings(df_raw, encoding_maps)
print(runtime_coverage.to_string(index=False))
print(f"Artefacto local: {map_path}")
print(f"Reporte local: {coverage_path}")

UPLOAD = False
if UPLOAD:
    target_key = f"{MODEL_PREFIX}/MODEL/artifacts_recuperados/{map_path.name}"
    s3.upload_file(str(map_path), BUCKET, target_key)
    print(f"Artefacto subido: s3://{BUCKET}/{target_key}")
else:
    print("UPLOAD=False: revise cobertura y coincidencia antes de publicar el artefacto.")

         variable  mapped_pct  unknown_count
    cod_ubigeo_cd       100.0              0
cod_sectorista_id       100.0              0
Artefacto local: artifacts_recuperados\encoding_maps_recuperados_202508.json
Reporte local: artifacts_recuperados\coverage_mapeos_202508.csv
UPLOAD=False: revise cobertura y coincidencia antes de publicar el artefacto.
